**Script created to automate the process of get the files from cloudfront and save at my S3 bucket.**

In [0]:
import boto3
import requests

# 1. Retrieve AWS credentials securely from Databricks Secrets
AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-keys", key="aws-access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-keys", key="aws-secret-key")
S3_BUCKET_NAME = "awae-yellow-taxi-case"

# URL for cloudfront raw files
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

# Create the S3 client once globally to reuse the connection pool across all files
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

In [0]:
def download_to_s3(client, base_url: str, file_name: str, s3_bucket: str, s3_folder: str):
    url = f"{base_url}{file_name}"
    s3_key = f"{s3_folder}/{file_name}"
    
    print(f"Starting download from: {url}")
    
    # Stream download to prevent driver memory overhead
    response = requests.get(url, stream=True)
    response.raise_for_status() 
    
    # Stream upload directly to AWS S3 using the shared client
    client.upload_fileobj(response.raw, s3_bucket, s3_key)
    print(f"File saved successfully: s3://{s3_bucket}/{s3_key}\n")

In [0]:
# 2. Configuration mapping for multi-source ingestion (Yellow and Green)
# Structure: "source_prefix": "destination_folder"
taxi_sources = {
    "yellow": "landing/yellow_tripdata",
    "green": "landing/green_tripdata"
}

# Target months to download (January to May 2023)
target_months = ["01", "02", "03", "04", "05"]

# 3. Dynamic execution loop (Eliminates duplicated code blocks)
for taxi_type, s3_destiny in taxi_sources.items():
    print(f"--- Starting Ingestion for {taxi_type.upper()} Taxi Data ---")
    
    for month in target_months:
        file_name = f"{taxi_type}_tripdata_2023-{month}.parquet"
        
        try:
            download_to_s3(
                client=s3_client,
                base_url=BASE_URL,
                file_name=file_name,
                s3_bucket=S3_BUCKET_NAME,
                s3_folder=s3_destiny
            )
        except Exception as e:
            print(f"File Processing Error - {file_name}: {e}")